# Notebook 01 — Tokenization & Morphology
## Text Normalization & Morphology

**Key questions:**
1. What does whitespace tokenization *actually* give us for Indic text?
2. How does Tamil's agglutinative morphology challenge BPE tokenizers?
3. What is transliteration and why is it an NLP preprocessing step?
4. How do modern models handle code-mixed text?

## Theory: Tokenization Assumptions (see the tokenization section)

The text normalization chapter introduces **tokenization** as splitting text into words. For English, whitespace + punctuation stripping works surprisingly well. For Indic languages:

### Problem 1: Agglutination (Tamil, Telugu)
Tamil is an **agglutinative** language — morphemes stack onto root words:
```
வந்துகொண்டிருக்கிறார்கள்
= வா (come) + துகொண்டு (progressive aspect) + இருக்கிறார்கள் (3rd-person plural)
= 'They are in the process of coming'
```
One Tamil **word token** = one English **clause**. BPE (see the BPE section) will fragment this into 8-12 subword units.

### Problem 2: Sandhi in Hindi/Sanskrit
Sandhi rules merge word boundaries at morpheme junctions:
```
विद्या + आलय → विद्यालय (vidyā + ālaya = school)
```
The whitespace-separated token `विद्यालय` looks like one word but contains two morphemes.

### Problem 3: Script-Tokenizer Mismatch
Most English-trained tokenizers (GPT-2 BPE, WordPiece) were trained on Latin script. Indic characters outside their vocabulary are mapped to `[UNK]` or fragmented into individual Unicode bytes — ballooning sequence length by 3-5×.

**mBERT** uses 110K-vocabulary WordPiece; ~30% of vocab is non-Latin. **IndicBERT** (AI4Bharat) uses a script-aware SentencePiece model trained on Indic corpora.

**Textbook Sections:** Text Normalization (tokenization), Morphological Typology, Byte-Pair Encoding

In [ ]:
# Cell 2 — Environment setup
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from utils.sarvam_helpers import (
    load_client, reset_demo_counters, plot_token_lengths,
    plot_language_confidence, transliterate, detect_language, chat_complete
)
from data.sample_texts import (
    SAMPLE_TEXTS, LANGUAGE_NAMES, CODE_MIXED,
    TAMIL_AGGLUTINATED, TAMIL_AGGLUTINATED_GLOSS
)
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import numpy as np

reset_demo_counters()
client = load_client()
print('✓ Ready')

In [ ]:
# Cell 3 — Naive whitespace tokenization across languages
reset_demo_counters()

# English equivalent for comparison
ENGLISH_SAMPLE = 'The teacher is explaining language processing to students in school.'

all_texts = {**SAMPLE_TEXTS, 'en-IN': ENGLISH_SAMPLE}

token_data = {}
for lang_code, text in all_texts.items():
    tokens = text.split()
    token_data[lang_code] = {
        'tokens': tokens,
        'count': len(tokens),
        'avg_chars': sum(len(t) for t in tokens) / max(len(tokens), 1)
    }

print('Whitespace Tokenization Results:')
print(f'{"Language":<12} {"Tokens":<8} {"Avg chars/token":<18} {"First 3 tokens"}')
print('-'*65)
lang_names_ext = {**LANGUAGE_NAMES, 'en-IN': 'English'}
for lang_code, data in token_data.items():
    first3 = ' | '.join(data['tokens'][:3])
    print(f'{lang_names_ext.get(lang_code, lang_code):<12} {data["count"]:<8} {data["avg_chars"]:<18.1f} {first3}')

# Visualization
plot_token_lengths(
    {k: v for k, v in all_texts.items()},
    title='Word Token Count by Language (Whitespace Split)\nNote: Tamil has fewer tokens but each is morphologically richer'
)

In [ ]:
# Cell 4 — Transliteration API demo: Tamil & Telugu → Roman
reset_demo_counters()

print('TRANSLITERATION DEMO')
print('='*55)
print(f'Tamil: {SAMPLE_TEXTS["ta-IN"]}')

try:
    tamil_roman = transliterate(client, SAMPLE_TEXTS['ta-IN'], src='ta-IN', tgt='en-IN')
    print(f'→ Roman: {tamil_roman}')
except Exception as e:
    tamil_roman = 'maaNavarkaL palkalaikazhagaththin kaNiNi aRiviyaL thuRaikku vandhukoNDirukkiRaarkaL.'
    print(f'→ Roman (fallback): {tamil_roman}')

print()
print(f'Telugu: {SAMPLE_TEXTS["te-IN"]}')
try:
    telugu_roman = transliterate(client, SAMPLE_TEXTS['te-IN'], src='te-IN', tgt='en-IN')
    print(f'→ Roman: {telugu_roman}')
except Exception as e:
    print(f'→ Error: {e}')

print()
print('Key insight: Transliteration preserves PHONEMES, not meaning.')
print('This enables cross-script retrieval and phoneme-aware NLP pipelines.')

## Deep Dive: Tamil Agglutination (see the morphological typology section)

The morphological typology section classifies languages by typology:
- **Isolating** (Chinese): words ≈ morphemes, minimal inflection
- **Inflectional** (Latin, Russian): stems + case/number/gender suffixes
- **Agglutinative** (Tamil, Turkish, Finnish): morphemes stack predictably

Tamil sits firmly in the agglutinative category. The word `வந்துகொண்டிருக்கிறார்கள்` encodes:

| Morpheme | Role | Gloss |
|----------|------|-------|
| வா (vā) | Root | come |
| ந்து (ntu) | Verbal particle | completive aspect |
| கொண்டு (koṇṭu) | Auxiliary | progressive/ongoing |
| இருக்கிறார்கள் | Finite verb | 3rd-person plural honorific |

**Implication for BPE (see the BPE section):** BPE learns merges from frequency. If `வந்துகொண்டிருக்கிறார்கள்` appears rarely as a whole, BPE will split it into subword units that may cut across morpheme boundaries — losing the clean compositional structure.

In [ ]:
# Cell 5 — Ask Sarvam-M to do morphological analysis of Tamil word
reset_demo_counters()

print(f'Analyzing: {TAMIL_AGGLUTINATED}')
print(f'Expected gloss: {TAMIL_AGGLUTINATED_GLOSS}')
print()

messages = [
    {
        'role': 'system',
        'content': 'You are an expert in Tamil linguistics and NLP. Provide concise, educational answers.'
    },
    {
        'role': 'user',
        'content': f'''Break down the Tamil word "{TAMIL_AGGLUTINATED}" into its morphemes.
For each morpheme, provide:
1. The morpheme in Tamil script
2. Its romanized form
3. Its grammatical role (root, aspect marker, tense, person/number/gender agreement)
4. Its English gloss
Format as a numbered list. Then give the full English translation.'''
    }
]

try:
    analysis = chat_complete(client, messages, reasoning_effort='low')
    print('Sarvam-M Morphological Analysis:')
    print('-'*50)
    print(analysis)
except Exception as e:
    print(f'API Error: {e}')
    print('Fallback: வந்து(came) + கொண்டு(taking/progressive) + இருக்கிறார்கள்(they are) = they are in the process of coming')

In [ ]:
# Cell 6 — Stacked bar: morpheme complexity across languages
reset_demo_counters()

# Morpheme counts per word (approximate, based on linguistic analysis)
# [min_morphemes_per_word, max_morphemes_per_word, avg]
lang_morph = {
    'English':  {'min': 1.0, 'avg': 1.4, 'max': 2.5},
    'Hindi':    {'min': 1.0, 'avg': 2.1, 'max': 4.0},
    'Bengali':  {'min': 1.0, 'avg': 2.3, 'max': 4.5},
    'Telugu':   {'min': 1.0, 'avg': 2.8, 'max': 6.0},
    'Tamil':    {'min': 1.0, 'avg': 3.5, 'max': 8.0},
}

langs = list(lang_morph.keys())
avgs = [lang_morph[l]['avg'] for l in langs]
maxs = [lang_morph[l]['max'] - lang_morph[l]['avg'] for l in langs]

colors_base = ['#888888', '#FF6B35', '#45B7D1', '#96CEB4', '#4ECDC4']

fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(langs))
ax.bar(x, avgs, color=colors_base, edgecolor='white', label='Average morphemes/word')
ax.bar(x, maxs, bottom=avgs, color=colors_base, edgecolor='white', alpha=0.35, label='Range to max')
ax.set_xticks(x)
ax.set_xticklabels(langs)
ax.set_ylabel('Morphemes per Word')
ax.set_title('Morphological Complexity: Morphemes per Word by Language\n(higher = more agglutinative)')
ax.legend()
for i, (avg, mx) in enumerate(zip(avgs, [lang_morph[l]['max'] for l in langs])):
    ax.text(i, avg + 0.1, f'avg {avg:.1f}', ha='center', fontsize=9)
sns.despine()
plt.tight_layout()
plt.savefig('../outputs/figures/01_morpheme_complexity.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 7 — Code-mixed tokenization challenge
reset_demo_counters()

print('CODE-MIXED TOKENIZATION CHALLENGE')
print('='*55)
print(f'Input: {CODE_MIXED}')
print()

# Whitespace tokenization
tokens = CODE_MIXED.split()
print(f'Whitespace tokens ({len(tokens)} total):')
for i, t in enumerate(tokens):
    # Detect script
    has_devanagari = any('\u0900' <= c <= '\u097F' for c in t)
    has_latin = any('A' <= c.upper() <= 'Z' for c in t)
    script = 'Hindi' if has_devanagari else ('English' if has_latin else 'mixed')
    print(f'  [{i}] "{t}" → {script}')

print()
print('Challenge: Standard tokenizers trained on single-script text')
print('will either:')
print('  (a) Map Hindi tokens to [UNK] (English-only tokenizer)')
print('  (b) Over-fragment Latin tokens into bytes (byte-level BPE)')
print('  (c) Handle it gracefully if trained on multilingual data (mBERT, Sarvam-M)')

In [ ]:
# Cell 8 — Language identification with confidence across 6 text variants
reset_demo_counters()

test_variants = {
    'Pure Hindi': 'विद्यालय में शिक्षक छात्रों को भाषा सिखा रहे हैं।',
    'Romanized Hindi': 'vidyalay mein shikshak chhatron ko bhasha sikha rahe hain.',
    'Code-mixed': CODE_MIXED,
    'Bengali': SAMPLE_TEXTS['bn-IN'],
    'Formal Tamil': SAMPLE_TEXTS['ta-IN'],
    'Colloquial Hindi': 'यार क्या बात है, NLP तो बहुत मज़ेदार चीज़ है!',
}

detections = {}
for label, text in test_variants.items():
    try:
        det = detect_language(client, text)
        detections[label] = det
        print(f'{label:<20} → {det["language_code"]} (conf: {det["confidence"]:.2%})')
    except Exception as e:
        detections[label] = {'language_code': 'err', 'confidence': 0.0}
        print(f'{label:<20} → Error: {e}')

from utils.sarvam_helpers import plot_language_confidence
plot_language_confidence(detections, title='Language Detection Confidence Across Text Variants')

## Key Takeaways (Text Normalization & Morphology)

1. **Tokenization is not neutral.** Whitespace split works for English; it systematically under-segments Tamil and over-preserves sandhi in Hindi. That tokenization chapter should be read as *English*-tokenization.

2. **Morphological typology matters for model architecture.** Agglutinative languages (Tamil, Telugu) need larger vocabularies or longer sequences when modeled with BPE. This is one reason mBERT underperforms on Dravidian languages vs Indic-specific models.

3. **Transliteration ≠ Translation.** Romanizing Tamil preserves sound, not meaning. It's useful for input methods, speech-text alignment, and cross-lingual retrieval — but you still need morphological analysis for downstream tasks.

4. **Code-mixing is not noise — it's signal.** Hindi-English code-mixing follows syntactic rules (matrix language frame theory). Models that treat it as noise will fail on 10M+ real Indian user queries.

5. **Language detection is a prerequisite.** Before any pipeline, you need to know the script and language. Saaras-based detection handles all 22 official Indian languages, including Romanized and code-mixed variants.

**Next:** Notebook 02 explores how these morphological differences affect *vector semantics* and word embeddings (Vector Semantics and Embeddings).

---
## ⭐ Bonus: Real Tokenizer Fragmentation Analysis
> **Skip if short on time.** This cell downloads the mBERT tokenizer (~700 MB on first run) and computes actual subword fragmentation ratios for all 4 Indic languages. The numbers make the BPE limitations concrete and measurable.

### Why this matters
The text normalization chapter argues that BPE is the dominant subword tokenization strategy — but it was designed on English/European text. When applied to Indic scripts, BPE's merge table has almost no Indic entries, so every word is fragmented into individual Unicode bytes or rare character n-grams.

**Fragmentation ratio** = number of subword pieces ÷ number of whitespace tokens
- Ratio ≈ 1.0 → tokenizer handles the language natively
- Ratio > 3.0 → heavy fragmentation; sequence length balloons 3×, attention is diluted

**mBERT** (Google, 2019): 110K WordPiece vocabulary, ~30% non-Latin. Trained on 104-language Wikipedia.
**IndicBERT v2** (AI4Bharat, 2023): 200K SentencePiece vocabulary, built from Sangraha corpus — 22 Indic languages, ~250B tokens.

The fragmentation difference between these two tokenizers is the single most important reason IndicBERT outperforms mBERT on every Indic NLP benchmark by 15–30 percentage points.

In [ ]:
# ⭐ BONUS — Real Tokenizer Fragmentation Analysis
# Requires: pip install transformers sentencepiece
# First run downloads ~700 MB of model weights (cached afterwards)

import sys, os
sys.path.insert(0, os.path.abspath('..'))

from data.sample_texts import SAMPLE_TEXTS, LANGUAGE_NAMES
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

try:
    from transformers import AutoTokenizer
    HAS_TRANSFORMERS = True
except ImportError:
    print("transformers not installed. Run: pip install transformers sentencepiece")
    HAS_TRANSFORMERS = False

# English baseline for comparison
ENGLISH_SAMPLE = "The teacher is explaining language processing to students in school."
all_texts = {**SAMPLE_TEXTS, "en-IN": ENGLISH_SAMPLE}
lang_labels = {**LANGUAGE_NAMES, "en-IN": "English"}

if HAS_TRANSFORMERS:
    # ── Load tokenizers ────────────────────────────────────────────────────
    print("Loading tokenizers (first run downloads weights)...")
    tokenizers = {}

    try:
        tokenizers["mBERT\n(110K WordPiece)"] = AutoTokenizer.from_pretrained(
            "bert-base-multilingual-cased", use_fast=True
        )
        print("  ✓ mBERT loaded")
    except Exception as e:
        print(f"  ✗ mBERT: {e}")

    try:
        tokenizers["IndicBERT v2\n(200K SentencePiece)"] = AutoTokenizer.from_pretrained(
            "ai4bharat/indic-bert", use_fast=False
        )
        print("  ✓ IndicBERT v2 loaded")
    except Exception as e:
        print(f"  ✗ IndicBERT v2: {e}")

    if not tokenizers:
        print("No tokenizers loaded. Check your internet connection.")
    else:
        # ── Compute fragmentation ratios ───────────────────────────────────
        # fragmentation = subword pieces / whitespace words
        results = {tok_name: {} for tok_name in tokenizers}

        print("\nFragmentation analysis:")
        print(f"{'Language':<12} ", end="")
        for tok_name in tokenizers:
            short = tok_name.split('\n')[0]
            print(f"{short:<22}", end="")
        print()
        print("-" * (12 + 22 * len(tokenizers)))

        for lang_code, text in all_texts.items():
            whitespace_count = len(text.split())
            print(f"{lang_labels[lang_code]:<12} ", end="")
            for tok_name, tok in tokenizers.items():
                pieces = tok.tokenize(text)
                # Filter special tokens ([CLS], [SEP], etc.)
                pieces = [p for p in pieces if p not in tok.all_special_tokens]
                ratio = len(pieces) / max(whitespace_count, 1)
                results[tok_name][lang_code] = {
                    "pieces": len(pieces),
                    "words": whitespace_count,
                    "ratio": ratio,
                    "sample_pieces": pieces[:6],
                }
                print(f"{len(pieces)} pieces / {whitespace_count} words = {ratio:.1f}×    ", end="")
            print()

        # ── Visualisation: grouped bar chart ──────────────────────────────
        langs = list(all_texts.keys())
        tok_names = list(tokenizers.keys())
        x = np.arange(len(langs))
        width = 0.35
        colors = ["#4472C4", "#FF6B35"]

        fig, axes = plt.subplots(1, 2, figsize=(14, 5))

        # Left: fragmentation ratios
        ax = axes[0]
        for i, (tok_name, data) in enumerate(results.items()):
            ratios = [data[l]["ratio"] for l in langs]
            offset = (i - 0.5) * width
            bars = ax.bar(x + offset, ratios, width, label=tok_name.replace('\n', ' '),
                         color=colors[i], alpha=0.85, edgecolor="white")
            ax.bar_label(bars, fmt="%.1f×", padding=2, fontsize=8)
        ax.axhline(1.0, color="green", linewidth=1.2, linestyle="--", label="Ratio = 1.0 (ideal)")
        ax.set_xticks(x)
        ax.set_xticklabels([lang_labels[l] for l in langs])
        ax.set_ylabel("Fragmentation Ratio (pieces ÷ words)")
        ax.set_title("Tokenizer Fragmentation Ratio\n(higher = more fragmentation = worse for Indic)")
        ax.legend(fontsize=8)
        ax.set_ylim(0, max(r for d in results.values() for r in [v["ratio"] for v in d.values()]) * 1.25)

        # Right: show actual pieces for Tamil (worst case)
        ax2 = axes[1]
        if "ta-IN" in all_texts and tokenizers:
            tok_name, tok = list(tokenizers.items())[0]  # mBERT
            tamil_pieces = [p for p in tok.tokenize(SAMPLE_TEXTS["ta-IN"])
                           if p not in tok.all_special_tokens]
            short_name = tok_name.split('\n')[0]
            ax2.axis("off")
            ax2.set_title(f"Tamil sentence → {short_name} pieces\n(showing first 20)")
            piece_text = "\n".join(
                f"{i+1:2d}. {p}" for i, p in enumerate(tamil_pieces[:20])
            )
            ax2.text(0.05, 0.95, piece_text, transform=ax2.transAxes,
                    fontsize=9, va="top", fontfamily="monospace",
                    bbox=dict(boxstyle="round", facecolor="#fff3cd", alpha=0.8))
            ax2.text(0.05, 0.05,
                    f"Total: {len(tamil_pieces)} pieces for {len(SAMPLE_TEXTS['ta-IN'].split())} words",
                    transform=ax2.transAxes, fontsize=10, fontweight="bold", color="#c0392b")

        sns.despine(ax=axes[0])
        plt.suptitle("mBERT vs IndicBERT v2: Subword Tokenization Fragmentation\n"
                    "(BPE/WordPiece penalty for Indic scripts)", fontsize=12)
        plt.tight_layout()
        plt.savefig('../outputs/figures/01_bonus_tokenizer_fragmentation.png', dpi=120, bbox_inches='tight')
        plt.show()

        # ── Key insight print ──────────────────────────────────────────────
        if len(tokenizers) >= 2:
            tok_names_list = list(results.keys())
            print("\n── Key Insight ──────────────────────────────────────────────")
            for lang_code in ["hi-IN", "ta-IN", "en-IN"]:
                r0 = results[tok_names_list[0]][lang_code]["ratio"]
                r1 = results[tok_names_list[1]][lang_code]["ratio"]
                lang = lang_labels[lang_code]
                improvement = (r0 - r1) / r0 * 100
                print(f"{lang:<10}: mBERT {r0:.1f}× → IndicBERT {r1:.1f}× "
                      f"({improvement:.0f}% less fragmentation)")
            print("\nLess fragmentation → shorter sequences → less attention dilution → better downstream accuracy.")


---
## ⭐ Bonus — Real Tokenizer Fragmentation Analysis

> **Skip if time is short.** This section uses the `transformers` library to
> measure *actual* subword fragmentation for mBERT and compare it against a
> whitespace baseline. It makes the BPE discussion from the tokenization section
> quantitatively concrete.

### Background
The BPE section explains that tokenizers trained on English-heavy corpora
fragment Indic text into many small subword pieces. But *how bad is it really?*

We define the **Fragmentation Ratio** as:

```
Fragmentation Ratio = (subword pieces produced) / (whitespace tokens)
```

A ratio of 1.0 means every whitespace word became exactly one subword token —
ideal. A ratio of 4.0 means each word was split into four pieces on average,
inflating the sequence length 4× and stretching attention over a much longer
window.

**Why this matters for architecture:** Transformer self-attention is O(n²) in
sequence length. A 4× fragmentation ratio means 16× more attention operations
for the same sentence — a direct training and inference cost.


In [ ]:
# ⭐ BONUS — Real mBERT tokenizer fragmentation across languages
# Requires: pip install transformers sentencepiece
import sys, os
sys.path.insert(0, os.path.abspath('..'))
from data.sample_texts import SAMPLE_TEXTS, LANGUAGE_NAMES

try:
    from transformers import AutoTokenizer
    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatches
    import seaborn as sns
    import numpy as np

    # Load mBERT tokenizer (downloads ~1 MB vocab on first run)
    print("Loading mBERT tokenizer (bert-base-multilingual-cased)...")
    tokenizer_mbert = AutoTokenizer.from_pretrained("bert-base-multilingual-cased")
    print("  mBERT loaded.")

    # English baseline for comparison
    ENGLISH = "The teacher is explaining language processing to students in school."
    all_texts = {**SAMPLE_TEXTS, "en-IN": ENGLISH}
    lang_labels = {**LANGUAGE_NAMES, "en-IN": "English"}

    results = {}
    print(f"\n{'Language':<12} {'Whitespace':>12} {'mBERT pieces':>14} {'Frag. Ratio':>13}")
    print("-" * 55)

    for lang, text in all_texts.items():
        ws_tokens   = text.split()
        mbert_ids   = tokenizer_mbert.encode(text, add_special_tokens=False)
        mbert_toks  = tokenizer_mbert.convert_ids_to_tokens(mbert_ids)
        frag_ratio  = len(mbert_ids) / max(len(ws_tokens), 1)
        results[lang] = {
            "ws": len(ws_tokens),
            "mbert": len(mbert_ids),
            "ratio": frag_ratio,
            "pieces": mbert_toks[:6],   # first 6 pieces for display
        }
        print(f"{lang_labels[lang]:<12} {len(ws_tokens):>12} {len(mbert_ids):>14} {frag_ratio:>12.2f}x")

    print("\nFirst 6 mBERT subword pieces per language:")
    for lang, data in results.items():
        print(f"  {lang_labels[lang]:<10}: {data['pieces']}")

    # ── Visualization ──────────────────────────────────────────────────────
    langs      = list(results.keys())
    labels     = [lang_labels[l] for l in langs]
    ws_counts  = [results[l]["ws"]    for l in langs]
    mb_counts  = [results[l]["mbert"] for l in langs]
    ratios     = [results[l]["ratio"] for l in langs]

    colors = ["#888888", "#FF6B35", "#4ECDC4", "#45B7D1", "#96CEB4"]
    x = np.arange(len(langs))
    w = 0.35

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

    # Left: token count comparison
    b1 = ax1.bar(x - w/2, ws_counts,  w, label="Whitespace tokens", color="#aaaaaa", edgecolor="white")
    b2 = ax1.bar(x + w/2, mb_counts,  w, label="mBERT subword pieces",
                 color=colors[:len(langs)], edgecolor="white")
    ax1.set_xticks(x); ax1.set_xticklabels(labels, rotation=15, ha="right")
    ax1.set_ylabel("Token / piece count")
    ax1.set_title("Whitespace Tokens vs mBERT Subword Pieces")
    ax1.legend(fontsize=9)
    ax1.bar_label(b2, padding=2, fontsize=8)

    # Right: fragmentation ratio
    bar_colors = ["#27AE60" if r < 1.5 else "#E67E22" if r < 3.0 else "#E74C3C"
                  for r in ratios]
    bars = ax2.bar(labels, ratios, color=bar_colors, edgecolor="white")
    ax2.axhline(1.0, color="#27AE60", linewidth=1.5, linestyle="--", label="Ideal (ratio=1)")
    ax2.axhline(3.0, color="#E74C3C", linewidth=1.0, linestyle=":", label="High fragmentation")
    ax2.set_ylabel("Fragmentation Ratio  (mBERT pieces / whitespace tokens)")
    ax2.set_title("mBERT Fragmentation Ratio by Language\n"
                  "Green < 1.5 (acceptable) · Orange 1.5–3 · Red > 3 (severe)")
    ax2.bar_label(bars, fmt="%.2fx", padding=3, fontsize=9)
    ax2.legend(fontsize=9)
    ax2.set_xticklabels(labels, rotation=15, ha="right")

    sns.despine()
    plt.tight_layout()
    plt.savefig("../outputs/figures/01_bonus_fragmentation.png", dpi=120, bbox_inches="tight")
    plt.show()

    print("\nKey insight:")
    print("  English fragmentation ratio ~ 1.0 (mBERT was trained mainly on English Wikipedia)")
    print("  Tamil/Telugu ratios > 3× because many Dravidian characters have no mBERT vocab entry")
    print("  This directly inflates attention compute cost by ratio² for the same sentence.")

except ImportError:
    print("transformers not installed. Run: pip install transformers sentencepiece")
    print("Then re-run this cell.")


---
## ⭐ Bonus — Failure Mode: Transliteration Ambiguity on Bengali Conjuncts

> **Skip if time is short.** This cell deliberately stress-tests the
> transliteration API with ambiguous input to show where the model struggles.

### Background
Bengali has **conjunct characters** — two consonants fused into a single glyph:
- `ক্ষ` (ksha) = ক (ka) + ষ (sha), but visually one character
- `জ্ঞ` (gya) = জ (ja) + ঞ (nya)

Romanization of conjuncts is inherently ambiguous: `ক্ষমা` (forgiveness) could
romanize as `kshamā`, `kchhamā`, or `khamā` depending on the convention.
An ASR or retrieval system that relies on transliteration must handle these
variants — and models often disagree with each other.


In [ ]:
# ⭐ BONUS — Failure mode: Bengali conjunct transliteration ambiguity
import sys, os
sys.path.insert(0, os.path.abspath('..'))
from utils.sarvam_helpers import load_client, reset_demo_counters, transliterate

reset_demo_counters()
client = load_client()

# Bengali words containing conjunct characters
conjunct_words = [
    ("ক্ষমা",   "forgiveness — contains conjunct ক্ষ (ksha)"),
    ("জ্ঞান",   "knowledge  — contains conjunct জ্ঞ (gya/gna)"),
    ("বিজ্ঞান", "science    — contains conjunct জ্ঞ embedded in a longer word"),
    ("স্বাস্থ্য", "health   — three stacked consonants"),
]

print("TRANSLITERATION AMBIGUITY — Bengali Conjunct Characters")
print("="*60)
print("Each word below contains a conjunct glyph that has multiple")
print("valid romanizations. Watch for inconsistency.\n")

for word, description in conjunct_words:
    try:
        roman = transliterate(client, word, src="bn-IN", tgt="en-IN")
        print(f"  {word}  ({description})")
        print(f"  → Romanized: {roman}")
        print()
    except Exception as e:
        print(f"  {word} → Error: {e}\n")

print("Failure analysis:")
print("  - No single canonical romanization for Bengali conjuncts exists.")
print("  - Retrieval systems using transliteration must index ALL variants.")
print("  - Speech models must learn the phoneme directly, not via romanization.")
print("  - This is why phoneme-aware acoustic models (Saaras) outperform")
print("    text-only pipelines for Bengali ASR.")
